# 20 Newsgroups + Qwen3 (Memory Optimized)

**GPU memory için optimize edilmiş versiyon**

In [ ]:
!pip install -q sentence-transformers datasets scikit-learn

In [ ]:
import json
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report
import gc

In [ ]:
# GPU memory temizle
torch.cuda.empty_cache()
gc.collect()
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

In [ ]:
# Dataset yükle - DAHA AZ SAMPLE
print("Loading 20 Newsgroups...")
dataset = load_dataset("SetFit/20_newsgroups", split="test")
dataset = dataset.shuffle(seed=42).select(range(1000))  # 2000 yerine 1000!

texts = dataset["text"]
true_labels = dataset["label"]

print(f"✅ {len(texts)} samples loaded")

In [ ]:
# Label descriptions
label_descriptions = [
    "This text discusses atheism, religious skepticism, secular humanism, or non-religious philosophy.",
    "This text discusses computer graphics, image processing, visualization, rendering, or graphical software.",
    "This text discusses Microsoft Windows operating system issues, tips, or questions.",
    "This text discusses IBM PC hardware, components, upgrades, or technical specifications.",
    "This text discusses Apple Macintosh hardware, components, or technical specifications.",
    "This text discusses X Window System, Unix graphical interface, or related software.",
    "This text is a for-sale advertisement, marketplace listing, or commercial offer.",
    "This text discusses automobiles, cars, driving, automotive technology, or vehicle maintenance.",
    "This text discusses motorcycles, bikes, riding, or motorcycle maintenance.",
    "This text discusses baseball, MLB, baseball players, games, or statistics.",
    "This text discusses hockey, NHL, hockey players, games, or ice hockey.",
    "This text discusses cryptography, encryption, security algorithms, or cryptographic systems.",
    "This text discusses electronics, circuits, electronic components, or electrical engineering.",
    "This text discusses medicine, medical conditions, healthcare, or medical research.",
    "This text discusses space, astronomy, space exploration, NASA, or astrophysics.",
    "This text discusses Christianity, Christian faith, Bible, or Christian theology.",
    "This text discusses gun politics, firearms, gun rights, or gun control debates.",
    "This text discusses Middle East politics, conflicts, or geopolitical issues in that region.",
    "This text discusses general political topics, political debates, or miscellaneous political issues.",
    "This text discusses general religious topics, interfaith dialogue, or miscellaneous religious matters."
]

In [ ]:
# Model yükle - FP16 ile
print("Loading Qwen/Qwen3-Embedding-4B with FP16...")
model = SentenceTransformer(
    "Qwen/Qwen3-Embedding-4B",
    trust_remote_code=True,
    model_kwargs={'torch_dtype': torch.float16}  # FP16 kullan!
)
if torch.cuda.is_available():
    model = model.half()  # Tüm modeli FP16'ya çevir
print("✅ Model loaded!")

In [ ]:
# Encode - ÇOK KÜÇÜK BATCH!
print("Encoding texts...")
text_embeddings = model.encode(
    texts,
    show_progress_bar=True,
    batch_size=4,  # 32 yerine 4!
    convert_to_numpy=True,
    normalize_embeddings=True
)

# Memory temizle
torch.cuda.empty_cache()
gc.collect()

print("Encoding labels...")
label_embeddings = model.encode(
    label_descriptions,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print(f"✅ Embeddings: {text_embeddings.shape}")

In [ ]:
# Model'i sil - memory boşalt
del model
torch.cuda.empty_cache()
gc.collect()
print("✅ Model cleared from memory")

In [ ]:
# Predictions
print("Computing predictions...")
similarities = np.dot(text_embeddings, label_embeddings.T)
predictions = np.argmax(similarities, axis=1)
confidences = np.max(similarities, axis=1)
print("✅ Done!")

In [ ]:
# Metrics
accuracy = accuracy_score(true_labels, predictions)
macro_f1 = f1_score(true_labels, predictions, average='macro')
weighted_f1 = f1_score(true_labels, predictions, average='weighted')
macro_precision = precision_score(true_labels, predictions, average='macro')
macro_recall = recall_score(true_labels, predictions, average='macro')
report = classification_report(true_labels, predictions, output_dict=True)

print("="*60)
print("SONUÇLAR (1000 samples)")
print("="*60)
print(f"Accuracy:        {accuracy*100:.2f}%")
print(f"Macro F1:        {macro_f1*100:.2f}%")
print(f"Weighted F1:     {weighted_f1*100:.2f}%")
print(f"Precision:       {macro_precision*100:.2f}%")
print(f"Recall:          {macro_recall*100:.2f}%")
print(f"Mean Confidence: {confidences.mean():.4f}")
print("="*60)

In [ ]:
# JSON sonuç
results = {
    "experiment_name": "SetFit_20_newsgroups_qwen3",
    "dataset_name": "SetFit/20_newsgroups",
    "num_samples": len(texts),
    "num_classes": 20,
    "accuracy": float(accuracy),
    "macro_f1": float(macro_f1),
    "weighted_f1": float(weighted_f1),
    "macro_precision": float(macro_precision),
    "macro_recall": float(macro_recall),
    "mean_confidence": float(confidences.mean()),
    "classification_report": {str(k): v for k, v in report.items()}
}

print("\n" + "="*60)
print("JSON ÇIKTISI (Kopyalayın!)")
print("="*60)
print(json.dumps(results, indent=2))
print("="*60)

In [ ]:
# Drive'a kaydet
from google.colab import drive
drive.mount('/content/drive')

with open('/content/drive/MyDrive/SetFit_20_newsgroups_qwen3_metrics.json', 'w') as f:
    json.dump(results, f, indent=2)

print("\n✅ Saved to Google Drive!")
print("\nŞimdi:")
print("1. JSON'u kopyala")
print("2. Local'de 'results/raw/SetFit_20_newsgroups_qwen3_metrics.json' olarak kaydet")
print("3. python generate_beautiful_plots.py çalıştır")
print("\n🎉 TAMAMLANDI!")